# Pretrain on MPOSE2021 -> Scratch/Fine-tune v2

서비스형 hard negative, detector-style augmentation, runtime-alignment metric을 반영한 v2 실험 노트북입니다.


In [1]:
import ctypes, glob, os
_nv = '/workspace/users/yijin/boot_env/.venv/lib/python3.12/site-packages/nvidia'
if os.path.isdir(_nv):
    for _so in sorted(glob.glob(f'{_nv}/*/lib/*.so.*')):
        try: ctypes.CDLL(_so, mode=ctypes.RTLD_GLOBAL)
        except OSError: pass

import json, sys, time
from argparse import Namespace
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

start = Path.cwd().resolve()
PROJECT_ROOT = None
for base in [start, *start.parents]:
    if (base / 'data' / 'reference_dances').exists() and (base / 'scripts' / 'prepare_mpose2021.py').exists():
        PROJECT_ROOT = base
        break
if PROJECT_ROOT is None:
    raise RuntimeError('project root not found')
sys.path.insert(0, str(PROJECT_ROOT))

import tensorflow as tf
from src.embedding.dataset_contrastive import TARGET_DANCES
from scripts.pretrain_scratch_mpose2021 import train as pretrain_train
from scripts.train_scratch_contrastive import train as scratch_train

print('PROJECT_ROOT:', PROJECT_ROOT)
print('TF version  :', tf.__version__)
print('GPU devices :', tf.config.list_physical_devices('GPU'))


I0000 00:00:1776753054.025606 2717360 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


PROJECT_ROOT: /workspace/users/yijin/boot_env/pjt
TF version  : 2.21.0
GPU devices : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
SEQUENCE_LENGTH = 30
TARGET_DANCES = list(TARGET_DANCES)

SIZES = {
    'small': {
        'embedding_dim': 32,
        'mlp_hidden': 64,
        'tcn_filters': 32, 'tcn_blocks': 3,
        'gcn_filters': 32, 'gcn_blocks': 2,
    },
    'base': {
        'embedding_dim': 64,
        'mlp_hidden': 128,
        'tcn_filters': 64, 'tcn_blocks': 4,
        'gcn_filters': 64, 'gcn_blocks': 3,
    },
}

def _size_kwargs(model, size):
    cfg = dict(SIZES[size])
    if model == 'mlp':
        cfg.setdefault('tcn_filters', 64); cfg.setdefault('tcn_blocks', 4)
        cfg.setdefault('gcn_filters', 64); cfg.setdefault('gcn_blocks', 3)
    elif model == 'tcn':
        cfg.setdefault('mlp_hidden', 128)
        cfg.setdefault('gcn_filters', 64); cfg.setdefault('gcn_blocks', 3)
    else:
        cfg.setdefault('mlp_hidden', 128)
        cfg.setdefault('tcn_filters', 64); cfg.setdefault('tcn_blocks', 4)
    return cfg


## 1. MPOSE2021 Pretrain v2


In [3]:
PRETRAIN_COMMON = dict(
    data_path=None,
    pose_extractor='movenet',
    split=1,
    output_dir='data/models/pretrain/mpose2021',
    sequence_length=SEQUENCE_LENGTH,
    dropout=0.15,
    epochs=100,
    batch_size=128,
    learning_rate=1e-3,
    min_learning_rate=1e-5,
    warmup_epochs=3,
    patience=10,
    temperature=0.1,
    runtime_jitter=0.01,
    augment_rot_deg=8.0,
    tcn_kernel=3,
    gcn_kernel=9,
    gcn_partition='distance',
    seed=42,
    verbose=2,
    prepare_if_missing=True,
)

PRETRAIN_VARIANTS = [
    {'enabled': True, 'model': m, 'size': s}
    for m in ('mlp', 'tcn', 'gcn')
    for s in ('small', 'base')
]

def run_pretrain_v2(variant):
    cfg = {**PRETRAIN_COMMON, 'model': variant['model']}
    cfg.update(_size_kwargs(variant['model'], variant['size']))
    dim = cfg['embedding_dim']
    cfg['model_name'] = f"mpose_{variant['model']}_{variant['size']}_e{dim}_final"
    return pretrain_train(Namespace(**cfg))

pretrain_paths = {}
pretrain_results = {}
for i, v in enumerate([x for x in PRETRAIN_VARIANTS if x.get('enabled', True)]):
    key = (v['model'], v['size'])
    print(f"\n[PRETRAIN {i+1}/{len(PRETRAIN_VARIANTS)}] {key}")
    res = run_pretrain_v2(v)
    pretrain_results[key] = res
    pretrain_paths[key] = res['paths']['weights']



[PRETRAIN 1/6] ('mlp', 'small')
[NORM] applying hip/torso normalization...
[DATA] train=12562, val=2867, T=30, J=12, C=2, num_classes=20


I0000 00:00:1776753067.746053 2717360 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 2331 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3060, pci bus id: 0000:0a:00.0, compute capability: 8.6


Model: "scratch_mlp_encoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ pose (InputLayer)               │ (None, 30, 12, 2)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_joints (Reshape)        │ (None, 30, 24)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ frame_dense_1 (Dense)           │ (None, 30, 64)         │         1,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ frame_bn_1 (BatchNormalization) │ (None, 30, 64)         │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ frame_relu_1 (Activation)       │ (None, 30, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ frame_drop_1 (Dropout)          │ (None, 30, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ frame_dense_2 (Dense)           │ (None, 30, 64)         │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ frame_bn_2 (BatchNormalization) │ (None, 30, 64)         │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ frame_relu_2 (Activation)       │ (None, 30, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ frame_drop_2 (Dropout)          │ (None, 30, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ temporal_pool                   │ (None, 64)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ head_hidden (Dense)             │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_raw (Dense)           │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Lambda)              │ (None, 32)             │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,408 (36.75 KB)

...

 Trainable params: 157,632 (615.75 KB)

 Non-trainable params: 896 (3.50 KB)

Epoch 1/100


I0000 00:00:1776753241.213066 2717928 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_142947__.105
I0000 00:00:1776753249.338153 2717926 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_144062__.12


98/98 - 14s - 145ms/step - class_retrieval_acc: 0.5054 - loss: 4.0863 - val_class_retrieval_acc: 0.7930 - val_loss: 4.7200 - learning_rate: 3.3333e-04
Epoch 2/100
98/98 - 2s - 18ms/step - class_retrieval_acc: 0.5848 - loss: 3.7504 - val_class_retrieval_acc: 0.8018 - val_loss: 4.4750 - learning_rate: 6.6667e-04
Epoch 3/100
98/98 - 2s - 18ms/step - class_retrieval_acc: 0.6049 - loss: 3.6505 - val_class_retrieval_acc: 0.8342 - val_loss: 4.2323 - learning_rate: 0.0010
Epoch 4/100
98/98 - 2s - 18ms/step - class_retrieval_acc: 0.6223 - loss: 3.5576 - val_class_retrieval_acc: 0.8555 - val_loss: 4.0944 - learning_rate: 0.0010
Epoch 5/100
98/98 - 2s - 18ms/step - class_retrieval_acc: 0.6346 - loss: 3.4898 - val_class_retrieval_acc: 0.8675 - val_loss: 3.9573 - learning_rate: 9.9974e-04
Epoch 6/100
98/98 - 2s - 18ms/step - class_retrieval_acc: 0.6435 - loss: 3.4480 - val_class_retrieval_acc: 0.8576 - val_loss: 3.9273 - learning_rate: 9.9896e-04
Epoch 7/100
98/98 - 2s - 17ms/step - class_retrieval

## 2. Scratch Contrastive v2


In [4]:
SCRATCH_COMMON_V2 = dict(
    data_dir='data/reference_dances',
    dances=TARGET_DANCES,
    output_dir='data/models/scratch',
    feature_dims=2,
    sequence_length=SEQUENCE_LENGTH,
    val_fraction=0.15,
    dropout=0.15,
    epochs=100,
    steps_per_epoch=300, # 200 300
    validation_steps=30,
    batch_size=256, # 128 256
    learning_rate=1e-3,
    min_learning_rate=1e-5,
    warmup_epochs=3,
    patience=10,
    temperature=0.1,
    triplet_margin=0.2,
    positive_jitter=2,
    negative_gap=90,
    false_negative_gap=4,
    hard_negative_min_gap=6,
    hard_negative_max_gap=24,
    hard_negative_prob=0.5,
    cross_song_prob=0.5,
    runtime_jitter=0.005,
    joint_dropout_prob=0.04,
    frame_hold_prob=0.05,
    temporal_warp_prob=0.25,
    temporal_warp_strength=0.15,
    eval_max_samples=128,
    eval_tolerance_frames=12,
    eval_candidate_stride=3,
    eval_user_runtime_jitter=0.01,
    eval_user_joint_dropout_prob=0.04,
    eval_user_frame_hold_prob=0.05,
    eval_user_temporal_warp_prob=0.25,
    eval_user_temporal_warp_strength=0.15,
    pretrained_weights=None,
    name_suffix='',
    seed=42,
    no_quantize=False,
    keep_checkpoint=False,
    verbose=2,
    tcn_kernel=3,
    gcn_kernel=9,
    gcn_partition='distance',
)

SCRATCH_VARIANTS_V2 = [
    {'enabled': True, 'model': m, 'size': s, 'loss': l}
    for m in ('mlp', 'tcn', 'gcn')
    for s in ('small', 'base')
    for l in ('infonce', 'triplet')
]

def run_scratch_v2(variant):
    cfg = {**SCRATCH_COMMON_V2, 'model': variant['model'], 'loss': variant['loss']}
    cfg.update(_size_kwargs(variant['model'], variant['size']))
    dim = cfg['embedding_dim']
    cfg['model_name'] = f"scratch_{variant['model']}_{variant['size']}_{variant['loss']}_e{dim}_final"
    return scratch_train(Namespace(**cfg))

scratch_results_v2 = {}
for i, v in enumerate([x for x in SCRATCH_VARIANTS_V2 if x.get('enabled', True)]):
    key = (v['model'], v['size'], v['loss'])
    print(f"\n[SCRATCH V2 {i+1}/{len(SCRATCH_VARIANTS_V2)}] {key}")
    scratch_results_v2[key] = run_scratch_v2(v)



[SCRATCH V2 1/12] ('mlp', 'small', 'infonce')
[404_dance] T=568, augments=30
[basic_movement] T=476, augments=30
[beginner_wave] T=679, augments=30
[cheerup_dance] T=724, augments=30
[hiphop_move] T=792, augments=30
[kpop_basic] T=7164, augments=30
[rasputin] T=3147, augments=30
[shuffle_dance] T=1538, augments=30
[DATA] 8 dances, feature_dims=2, val_fraction=0.15


Model: "scratch_mlp_encoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ pose (InputLayer)               │ (None, 30, 12, 2)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_joints (Reshape)        │ (None, 30, 24)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ frame_dense_1 (Dense)           │ (None, 30, 64)         │         1,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ frame_bn_1 (BatchNormalization) │ (None, 30, 64)         │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ frame_relu_1 (Activation)       │ (None, 30, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ frame_drop_1 (Dropout)          │ (None, 30, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ frame_dense_2 (Dense)           │ (None, 30, 64)         │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ frame_bn_2 (BatchNormalization) │ (None, 30, 64)         │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ frame_relu_2 (Activation)       │ (None, 30, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ frame_drop_2 (Dropout)          │ (None, 30, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ temporal_pool                   │ (None, 64)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ head_hidden (Dense)             │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_raw (Dense)           │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Lambda)              │ (None, 32)             │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,408 (36.75 KB)

 Trainable params: 9,152 (35.75 KB)

...

Epoch 1/100


I0000 00:00:1776761300.197744 2717930 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_1073473__.237
I0000 00:00:1776761332.299678 2717924 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_1076186__.12


300/300 - 42s - 139ms/step - loss: 0.0195 - pos_margin: 0.7313 - val_loss: 0.0044 - val_pos_margin: 0.8414 - val_violation_rate: 0.0400 - violation_rate: 0.1540 - learning_rate: 3.3333e-04
Epoch 2/100
300/300 - 26s - 86ms/step - loss: 0.0061 - pos_margin: 0.7978 - val_loss: 0.0054 - val_pos_margin: 0.7901 - val_violation_rate: 0.0488 - violation_rate: 0.0567 - learning_rate: 6.6667e-04
Epoch 3/100
300/300 - 26s - 86ms/step - loss: 0.0043 - pos_margin: 0.7930 - val_loss: 0.0065 - val_pos_margin: 0.7036 - val_violation_rate: 0.0609 - violation_rate: 0.0430 - learning_rate: 0.0010
Epoch 4/100
300/300 - 26s - 86ms/step - loss: 0.0031 - pos_margin: 0.8049 - val_loss: 0.0062 - val_pos_margin: 0.6603 - val_violation_rate: 0.0668 - violation_rate: 0.0316 - learning_rate: 0.0010
Epoch 5/100
300/300 - 26s - 86ms/step - loss: 0.0024 - pos_margin: 0.8120 - val_loss: 0.0040 - val_pos_margin: 0.7035 - val_violation_rate: 0.0415 - violation_rate: 0.0251 - learning_rate: 9.9974e-04
Epoch 6/100
300/300

I0000 00:00:1776762210.187165 2717360 devices.cc:67] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 1
I0000 00:00:1776762210.187255 2717360 single_machine.cc:376] Starting new session
I0000 00:00:1776762210.193088 2717360 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 2331 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3060, pci bus id: 0000:0a:00.0, compute capability: 8.6
W0000 00:00:1776762210.368619 2717360 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1776762210.368639 2717360 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1776762210.427719 2717360 flatbuffer_export.cc:4302] Estimated count of arithmetic ops: 111.899 M  ops, equivalently 55.949 M  MACs
I0000 00:00:1776762210.785569 2717929 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_1137355__.3
I0000 00:00:1776762211.873319 2717927 dot_merger.cc:481] Merging Dots in computation: a

[SAVE] encoder  : /workspace/users/yijin/boot_env/pjt/data/models/scratch/scratch_gcn_base_triplet_e64_final_encoder.keras
[SAVE] tflite   : /workspace/users/yijin/boot_env/pjt/data/models/scratch/scratch_gcn_base_triplet_e64_final.tflite (188.4 KiB)
[BEST] summary  : {'monitor': 'val_loss', 'restore_best_weights': True, 'epochs_ran': 35, 'best_epoch': 25, 'best_val_loss': 0.0018119695596396923, 'best_epoch_loss': 0.0009627273539081216}
[METRIC] smoke  : {'cos_same_t_mean': 0.9857122898101807, 'cos_same_song_far_t_mean': 0.10042137652635574, 'cos_other_song_mean': 0.05765114724636078, 'within_dance_margin_mean': 0.8852909207344055, 'cross_dance_margin_mean': 0.9280611872673035}
[METRIC] align  : {'samples': 128, 'target_cosine_mean': 0.9773732423782349, 'best_wrong_cosine_mean': 0.8830806016921997, 'target_margin_mean': 0.09429260343313217, 'target_rank_mean': 1.046875, 'top1_acc': 0.9765625, 'top3_acc': 0.9921875}
[SAVE] metadata : /workspace/users/yijin/boot_env/pjt/data/models/scrat

## 3. Fine-tune v2 (same trainer, pretrained weights loaded)


In [ ]:
PRETRAIN_DIR = PROJECT_ROOT / 'data/models/pretrain/mpose2021'
pretrain_paths = {}
for m in ('mlp', 'tcn', 'gcn'):
    for s in ('small', 'base'):
        dim = SIZES[s]['embedding_dim']
        p = PRETRAIN_DIR / f'mpose_{m}_{s}_e{dim}_final.weights.h5'
        if p.exists():
            pretrain_paths[(m, s)] = str(p)
        else:
            print(f'[MISSING] {p}')
print(f'pretrain_paths: {len(pretrain_paths)}/6 loaded')

In [5]:
FINETUNE_COMMON_V2 = dict(
    data_dir='data/reference_dances',
    dances=TARGET_DANCES,
    output_dir='data/models/embedding',
    feature_dims=2,
    sequence_length=SEQUENCE_LENGTH,
    val_fraction=0.15,
    dropout=0.15,
    epochs=80,
    steps_per_epoch=300,
    validation_steps=30,
    batch_size=256,
    learning_rate=1e-4,
    min_learning_rate=1e-6,
    warmup_epochs=1,
    patience=8,
    temperature=0.1,
    triplet_margin=0.2,
    positive_jitter=2,
    negative_gap=90,
    false_negative_gap=4,
    hard_negative_min_gap=6,
    hard_negative_max_gap=24,
    hard_negative_prob=0.5,
    cross_song_prob=0.5,
    runtime_jitter=0.005,
    joint_dropout_prob=0.04,
    frame_hold_prob=0.05,
    temporal_warp_prob=0.25,
    temporal_warp_strength=0.15,
    eval_max_samples=128,
    eval_tolerance_frames=12,
    eval_candidate_stride=3,
    eval_user_runtime_jitter=0.01,
    eval_user_joint_dropout_prob=0.04,
    eval_user_frame_hold_prob=0.05,
    eval_user_temporal_warp_prob=0.25,
    eval_user_temporal_warp_strength=0.15,
    pretrained_weights=None,
    name_suffix='',
    seed=42,
    no_quantize=False,
    keep_checkpoint=False,
    verbose=2,
    tcn_kernel=3,
    gcn_kernel=9,
    gcn_partition='distance',
)

FINETUNE_VARIANTS_V2 = [
    {'enabled': True, 'model': m, 'size': s, 'loss': l}
    for m in ('mlp', 'tcn', 'gcn')
    for s in ('small', 'base')
    for l in ('infonce', 'triplet')
]

def run_finetune_v2(variant, pretrained_path):
    cfg = {**FINETUNE_COMMON_V2, 'model': variant['model'], 'loss': variant['loss']}
    cfg.update(_size_kwargs(variant['model'], variant['size']))
    dim = cfg['embedding_dim']
    cfg['model_name'] = f"finetune_{variant['model']}_{variant['size']}_{variant['loss']}_e{dim}_final"
    cfg['pretrained_weights'] = pretrained_path
    return scratch_train(Namespace(**cfg))

finetune_results_v2 = {}
for i, v in enumerate([x for x in FINETUNE_VARIANTS_V2 if x.get('enabled', True)]):
    key = (v['model'], v['size'], v['loss'])
    pretrained_key = (v['model'], v['size'])
    if pretrained_key not in pretrain_paths:
        print(f"[SKIP] {key}: missing pretrain weights")
        continue
    print(f"\n[FINETUNE V2 {i+1}/{len(FINETUNE_VARIANTS_V2)}] {key}")
    finetune_results_v2[key] = run_finetune_v2(v, pretrain_paths[pretrained_key])



[FINETUNE V2 1/12] ('mlp', 'small', 'infonce')
[404_dance] T=568, augments=30
[basic_movement] T=476, augments=30
[beginner_wave] T=679, augments=30
[cheerup_dance] T=724, augments=30
[hiphop_move] T=792, augments=30
[kpop_basic] T=7164, augments=30
[rasputin] T=3147, augments=30
[shuffle_dance] T=1538, augments=30
[DATA] 8 dances, feature_dims=2, val_fraction=0.15
[FINETUNE] loaded pretrained weights from /workspace/users/yijin/boot_env/pjt/data/models/pretrain/mpose2021/mpose_mlp_small_e32_final.weights.h5


Model: "scratch_mlp_encoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ pose (InputLayer)               │ (None, 30, 12, 2)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_joints (Reshape)        │ (None, 30, 24)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ frame_dense_1 (Dense)           │ (None, 30, 64)         │         1,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ frame_bn_1 (BatchNormalization) │ (None, 30, 64)         │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ frame_relu_1 (Activation)       │ (None, 30, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ frame_drop_1 (Dropout)          │ (None, 30, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ frame_dense_2 (Dense)           │ (None, 30, 64)         │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ frame_bn_2 (BatchNormalization) │ (None, 30, 64)         │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ frame_relu_2 (Activation)       │ (None, 30, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ frame_drop_2 (Dropout)          │ (None, 30, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ temporal_pool                   │ (None, 64)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ head_hidden (Dense)             │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_raw (Dense)           │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Lambda)              │ (None, 32)             │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,408 (36.75 KB)

 Trainable params: 9,152 (35.75 KB)

...

Epoch 1/80


I0000 00:00:1776771453.100121 2717926 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_2184758__.237
I0000 00:00:1776771484.786372 2717927 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_2187471__.12


300/300 - 41s - 136ms/step - loss: 0.0242 - pos_margin: 0.5886 - val_loss: 0.0105 - val_pos_margin: 0.8078 - val_violation_rate: 0.0922 - violation_rate: 0.1793 - learning_rate: 1.0000e-04
Epoch 2/80
300/300 - 26s - 86ms/step - loss: 0.0135 - pos_margin: 0.7532 - val_loss: 0.0060 - val_pos_margin: 0.8853 - val_violation_rate: 0.0527 - violation_rate: 0.1057 - learning_rate: 1.0000e-04
Epoch 3/80
300/300 - 26s - 86ms/step - loss: 0.0101 - pos_margin: 0.7859 - val_loss: 0.0045 - val_pos_margin: 0.9230 - val_violation_rate: 0.0397 - violation_rate: 0.0838 - learning_rate: 9.9961e-05
Epoch 4/80
300/300 - 26s - 86ms/step - loss: 0.0083 - pos_margin: 0.8009 - val_loss: 0.0047 - val_pos_margin: 0.9070 - val_violation_rate: 0.0419 - violation_rate: 0.0699 - learning_rate: 9.9844e-05
Epoch 5/80
300/300 - 26s - 86ms/step - loss: 0.0071 - pos_margin: 0.8072 - val_loss: 0.0037 - val_pos_margin: 0.9198 - val_violation_rate: 0.0346 - violation_rate: 0.0620 - learning_rate: 9.9648e-05
Epoch 6/80
300/

I0000 00:00:1776772183.960289 2717360 devices.cc:67] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 1
I0000 00:00:1776772183.960416 2717360 single_machine.cc:376] Starting new session
I0000 00:00:1776772183.966282 2717360 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 2331 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3060, pci bus id: 0000:0a:00.0, compute capability: 8.6
W0000 00:00:1776772184.107507 2717360 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1776772184.107527 2717360 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1776772184.165777 2717360 flatbuffer_export.cc:4302] Estimated count of arithmetic ops: 111.899 M  ops, equivalently 55.949 M  MACs
I0000 00:00:1776772184.511988 2717931 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_2237049__.3
I0000 00:00:1776772186.987820 2717931 dot_merger.cc:481] Merging Dots in computation: a

[SAVE] encoder  : /workspace/users/yijin/boot_env/pjt/data/models/embedding/finetune_gcn_base_triplet_e64_final_encoder.keras
[SAVE] tflite   : /workspace/users/yijin/boot_env/pjt/data/models/embedding/finetune_gcn_base_triplet_e64_final.tflite (188.4 KiB)
[BEST] summary  : {'monitor': 'val_loss', 'restore_best_weights': True, 'epochs_ran': 28, 'best_epoch': 20, 'best_val_loss': 0.002775064669549465, 'best_epoch_loss': 0.0026955718640238047}
[METRIC] smoke  : {'cos_same_t_mean': 0.9837517738342285, 'cos_same_song_far_t_mean': 0.03756078705191612, 'cos_other_song_mean': -0.007834846153855324, 'within_dance_margin_mean': 0.9461909532546997, 'cross_dance_margin_mean': 0.9915866255760193}
[METRIC] align  : {'samples': 128, 'target_cosine_mean': 0.9730536341667175, 'best_wrong_cosine_mean': 0.9348206520080566, 'target_margin_mean': 0.03823288157582283, 'target_rank_mean': 1.1875, 'top1_acc': 0.8671875, 'top3_acc': 0.984375}
[SAVE] metadata : /workspace/users/yijin/boot_env/pjt/data/models/e

## 4. Compare v2 metrics


In [4]:
def _final(history, keys):
    for k in keys:
        if k in history and history[k]:
            return history[k][-1]
    return float('nan')

rows = []
for v in FINETUNE_VARIANTS_V2:
    key = (v['model'], v['size'], v['loss'])
    for regime, res in [('scratch_v2', scratch_results_v2.get(key)), ('finetune_v2', finetune_results_v2.get(key))]:
        if res is None:
            continue
        hist = res.get('history', {})
        summary = res.get('training_summary', {})
        smoke = res.get('smoke_metrics', {})
        align = res.get('runtime_alignment_metrics', {})
        rows.append({
            'model': v['model'],
            'size': v['size'],
            'loss': v['loss'],
            'regime': regime,
            'epochs_run': len(hist.get('loss', [])),
            'best_val_loss': summary.get('best_val_loss'),
            'best_epoch': summary.get('best_epoch'),
            'within_margin': smoke.get('within_dance_margin_mean'),
            'cross_margin': smoke.get('cross_dance_margin_mean'),
            'runtime_top1': align.get('top1_acc'),
            'runtime_top3': align.get('top3_acc'),
            'runtime_margin': align.get('target_margin_mean'),
            'runtime_rank_mean': align.get('target_rank_mean'),
        })
summary_v2 = pd.DataFrame(rows).sort_values(['model', 'size', 'loss', 'regime'])
summary_v2


NameError: name 'FINETUNE_VARIANTS_V2' is not defined

In [ ]:
fig, axes = plt.subplots(len(FINETUNE_VARIANTS_V2), 2, figsize=(14, 2.2 * len(FINETUNE_VARIANTS_V2)))
if len(FINETUNE_VARIANTS_V2) == 1:
    axes = axes[None, :]

for row, v in enumerate(FINETUNE_VARIANTS_V2):
    key = (v['model'], v['size'], v['loss'])
    rs = scratch_results_v2.get(key)
    rf = finetune_results_v2.get(key)
    title = f"{v['model']}/{v['size']}/{v['loss']}"

    ax = axes[row, 0]
    if rs is not None:
        h = rs['history']
        ax.plot(h.get('loss', []), 'b-', label='scratch train', alpha=0.85)
        if 'val_loss' in h:
            ax.plot(h['val_loss'], 'b--', alpha=0.55, label='scratch val')
    if rf is not None:
        h = rf['history']
        ax.plot(h.get('loss', []), 'r-', label='finetune train', alpha=0.85)
        if 'val_loss' in h:
            ax.plot(h['val_loss'], 'r--', alpha=0.55, label='finetune val')
    ax.set_title(f'{title} - loss')
    ax.set_xlabel('epoch')
    ax.legend(fontsize=7, loc='upper right')

    ax = axes[row, 1]
    metric = 'retrieval_acc' if v['loss'] == 'infonce' else 'pos_margin'
    val_metric = f'val_{metric}'
    if rs is not None:
        h = rs['history']
        if metric in h:
            ax.plot(h[metric], 'b-', label='scratch', alpha=0.85)
        if val_metric in h:
            ax.plot(h[val_metric], 'b--', alpha=0.55)
    if rf is not None:
        h = rf['history']
        if metric in h:
            ax.plot(h[metric], 'r-', label='finetune', alpha=0.85)
        if val_metric in h:
            ax.plot(h[val_metric], 'r--', alpha=0.55)
    ax.set_title(f'{title} - {metric}')
    ax.set_xlabel('epoch')
    ax.legend(fontsize=7, loc='lower right')

plt.tight_layout()
plt.show()


## 5. Optional: exported TFLite runtime-alignment check


In [ ]:
from scripts.eval_runtime_alignment import evaluate_runtime_alignment

example_variant = {'model': 'tcn', 'size': 'base', 'loss': 'infonce'}
example_name = f"finetune_{example_variant['model']}_{example_variant['size']}_{example_variant['loss']}_e{SIZES[example_variant['size']]['embedding_dim']}_final"
args = Namespace(
    kind='embedding',
    model_name=example_name,
    model_dir='data/models/embedding',
    model_path=None,
    data_dir='data/reference_dances',
    dances=TARGET_DANCES,
    feature_dims=2,
    sequence_length=SEQUENCE_LENGTH,
    candidate_stride=3,
    tolerance_frames=12,
    samples=128,
    seed=42,
    user_runtime_jitter=0.01,
    user_joint_dropout_prob=0.04,
    user_frame_hold_prob=0.05,
    user_temporal_warp_prob=0.25,
    user_temporal_warp_strength=0.15,
    output=None,
)
evaluate_runtime_alignment(args)
